# mm-pipeline — end-to-end example

This notebook shows the mm-pipeline run end-to-end on a small synthetic mother-machine trench bundled with the package (under `example_data/`, generated with [SyMBac](https://github.com/georgeoshardo/SyMBac)). Two examples on the same trench:

- **Example A** — DP baseline (no scorer): `candidates → featurise → qa → tracks`
- **Example B** — Classifier-driven: `candidates → featurise → score → qa → tracks`

After each example we plot two views of the reconstructed lineage:

1. A **mother-cell length time series** that follows the cell at the closed end across many generations.
2. A **full lineage swimlane** showing every track's spatial position over time with division connectors and the mother lineage highlighted.

## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from mm_pipeline.config.schemas import DatasetSpec, QAConfig
from mm_pipeline.features.pairwise import build_feature_dataframe
from mm_pipeline.runners import (
    run_candidates,
    run_featurise,
    run_qa,
    run_score,
    run_train_scorer,
)

EXAMPLE_DATA = Path("../example_data")

ds = DatasetSpec(
    dataset_id="symbac_vid2",
    labels_dir=EXAMPLE_DATA / "labels",
    gt_tracks_csv=EXAMPLE_DATA / "gt_tracks.csv",
    gt_divisions_csv=EXAMPLE_DATA / "gt_divisions.csv",
    axis="y",
    open_end="high",
)
specs = [ds]

print(f"Dataset: {ds.dataset_id}")
print(f"  labels: {ds.labels_dir}")
print(f"  GT tracks: {ds.gt_tracks_csv}")
print(f"  GT divisions: {ds.gt_divisions_csv}")

## 2. Visualisation helpers

Two groups of helpers:

- **Mother-lineage tracing** (`build_lineage_tree`, `trace_branch`, `assemble_series`, `_pick_closed_daughter`): identify the chain of `track_id`s forming the mother lineage. At each division they follow the daughter at the closed end.
- **Plotting** (`plot_mother_length_series`, `plot_lineage_swimlane`): turn `tracks_df` + `divisions_df` into the two visualisations we use after each example.

In [ ]:
def build_lineage_tree(tracks_df, div_df, open_end):
    """Full lineage tree from tracks + division events."""
    children_map, parent_map, t_div_map = {}, {}, {}
    for _, r in div_df.iterrows():
        m = int(r["mother_track_id"])
        d1 = int(r["d1_track_id"])
        d2 = int(r["d2_track_id"])
        children_map[m] = (d1, d2)
        parent_map[d1] = m
        parent_map[d2] = m
        t_div_map[m] = int(r["t_div"])

    birth, frame_range = {}, {}
    for tid, seg in tracks_df.groupby("track_id"):
        tid = int(tid)
        row = seg.loc[seg["t"].idxmin()]
        birth[tid] = {"t": int(row["t"]), "y": float(row["y"])}
        frame_range[tid] = (int(seg["t"].min()), int(seg["t"].max()))

    return {
        "children_map": children_map,
        "parent_map": parent_map,
        "t_div_map": t_div_map,
        "birth": birth,
        "frame_range": frame_range,
        "open_end": open_end,
    }


def _pick_closed_daughter(d1, d2, y1, y2, open_end):
    """Daughter nearest the closed end = the continuing mother."""
    if y1 is None or y2 is None:
        return d1
    if open_end == "high":
        return d1 if y1 < y2 else d2
    return d1 if y1 > y2 else d2


def trace_branch(tree, policy="mother", start=None):
    """Ordered list of track_ids forming one branch (the mother lineage)."""
    if policy != "mother":
        raise NotImplementedError(f"policy={policy!r} not implemented")
    birth = tree["birth"]
    children = tree["children_map"]
    open_end = tree["open_end"]

    if start is None:
        t0 = min(b["t"] for b in birth.values())
        cands = [(tid, b["y"]) for tid, b in birth.items() if b["t"] == t0]
        chooser = min if open_end == "high" else max
        start = chooser(cands, key=lambda kv: kv[1])[0]

    branch, seen = [], set()
    cur = int(start)
    while cur is not None and cur not in seen:
        if cur not in birth:
            break
        branch.append(cur)
        seen.add(cur)
        if cur not in children:
            break
        d1, d2 = children[cur]
        y1 = birth.get(d1, {}).get("y")
        y2 = birth.get(d2, {}).get("y")
        cur = _pick_closed_daughter(d1, d2, y1, y2, open_end)
    return branch


def assemble_series(tracks_df, branch, feature="axis_len"):
    """Concatenated time-series for the branch; one segment_id per cell cycle."""
    parts = []
    for seg_idx, tid in enumerate(branch):
        seg = tracks_df[tracks_df["track_id"] == tid][["t", feature]].copy()
        seg = seg.sort_values("t")
        seg["track_id"] = tid
        seg["segment_id"] = seg_idx
        parts.append(seg)
    return (
        pd.concat(parts, ignore_index=True)
        .sort_values(["t", "segment_id"])
        .reset_index(drop=True)
    )


def plot_mother_length_series(tracks_df, divisions_df, ds, title_suffix=""):
    """Plot mother-lineage axis_len vs frame, with division markers."""
    tree = build_lineage_tree(tracks_df, divisions_df, ds.open_end)
    branch = trace_branch(tree, policy="mother")
    series = assemble_series(tracks_df, branch, feature="axis_len")
    div_ts = sorted(tree["t_div_map"][tid] for tid in branch if tid in tree["t_div_map"])

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(series["t"], series["axis_len"], color="#1f77b4", lw=1.0,
            marker="o", ms=4, alpha=0.9, label="cell length")
    for i, td in enumerate(div_ts):
        ax.axvline(td, color="#d62728", ls="--", lw=1.0, alpha=0.4,
                   label="division" if i == 0 else None)
    ax.set_xlabel("frame t")
    ax.set_ylabel("cell length (px)")
    ax.set_title(f"{ds.dataset_id} — mother-lineage length "
                 f"({len(branch)} cell cycles){title_suffix}")
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()
    return series, branch, div_ts


def plot_lineage_swimlane(tracks_df, divisions_df, ds, mother_branch=None, title_suffix=""):
    """Full lineage swimlane: each track plotted as (t, y) with division
    connectors. The mother lineage (if provided) is highlighted in red.
    Y-axis is inverted so the closed end of the trench is at the top.
    """
    mother_set = set(int(t) for t in (mother_branch or []))

    endpoints = {}
    for tid, seg in tracks_df.groupby("track_id"):
        seg = seg.sort_values("t")
        first = seg.iloc[0]
        last = seg.iloc[-1]
        endpoints[int(tid)] = (
            (int(first["t"]), float(first["y"])),
            (int(last["t"]), float(last["y"])),
        )

    fig, ax = plt.subplots(figsize=(12, 5))
    for tid, seg in tracks_df.groupby("track_id"):
        seg = seg.sort_values("t")
        is_mother = int(tid) in mother_set
        ax.plot(
            seg["t"], seg["y"],
            color="#d62728" if is_mother else "#666666",
            lw=1.8 if is_mother else 0.6,
            alpha=0.95 if is_mother else 0.55,
        )

    for _, row in divisions_df.iterrows():
        m = int(row["mother_track_id"])
        d1 = int(row["d1_track_id"])
        d2 = int(row["d2_track_id"])
        if m not in endpoints:
            continue
        m_last = endpoints[m][1]
        for d in (d1, d2):
            if d not in endpoints:
                continue
            d_first = endpoints[d][0]
            in_mother = (m in mother_set) and (d in mother_set)
            ax.plot(
                [m_last[0], d_first[0]],
                [m_last[1], d_first[1]],
                color="#d62728" if in_mother else "#aaaaaa",
                lw=1.2 if in_mother else 0.5,
                ls="--",
                alpha=0.7 if in_mother else 0.4,
            )

    if ds.open_end == "high":
        ax.invert_yaxis()
        closed_loc = "top"
    else:
        closed_loc = "bottom"

    ax.set_xlabel("frame t")
    ax.set_ylabel(f"cell y position (px) — closed end at {closed_loc}")
    ax.set_title(
        f"{ds.dataset_id} — lineage swimlane "
        f"({tracks_df['track_id'].nunique()} tracks, "
        f"{len(divisions_df)} divisions){title_suffix}"
    )
    if mother_branch:
        ax.plot([], [], color="#d62728", lw=1.8, label="mother lineage")
        ax.plot([], [], color="#666666", lw=0.6, label="other tracks")
        ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

## 3. Prep — train a candidate-plausibility scorer

**One-off step.** In production you'd be given a trained scorer (a `.joblib` file). We train one here so the notebook is self-contained, using the bundled trench's ground truth via `build_feature_dataframe` (the standard training-data preparation function).

In [ ]:
# Build labelled training data. GT injection adds the ground-truth candidate
# to each pair when DP missed it, so every pair has at least one positive.
train_df, failures_df = build_feature_dataframe(
    specs,
    gt_mode="saved",
    include_gt_if_missing=True,
    store_ops=True,
    top_k_candidates=16,
)

n_pairs = train_df["pair_id"].nunique()
n_correct = int(train_df["is_correct"].fillna(False).sum())
print(f"Training data: {len(train_df):,} candidates across {n_pairs:,} pairs")
print(f"  positive (is_correct=True): {n_correct:,}")
print(f"  failures: {len(failures_df)}")

In [ ]:
# Fit an in-memory scorer. out_path=None => no joblib is written; the
# FittedScorer lives entirely in this kernel.
train_result = run_train_scorer(
    train_df,
    model_name="random_forest_balanced",
    feature_subset="all_features",
    out_path=None,
)
scorer = train_result.fitted_scorer
print(f"Fitted {scorer.model_name} on {len(scorer.feature_cols)} features.")

## 4. Example A — DP baseline (no scorer)

Simplest deployment: trust DP's top-1 candidate at every pair. Default `QAConfig()` selects this path. No model needed.

In [ ]:
# Stage 1: candidates from labels.
cand = run_candidates(ds, top_k=16)
print(f"Generated {len(cand.candidates_df):,} candidates "
      f"across {cand.candidates_df['pair_id'].nunique():,} frame pairs.")

In [ ]:
# Stage 2: 14 pairwise features per candidate.
feat = run_featurise(ds, candidates=cand.candidates_df)
print(f"Featurised: {len(feat.features_df):,} rows, "
      f"{len(feat.features_df.columns)} columns.")

In [ ]:
# Stage 3: QA workflow + lineage reconstruction (DP baseline = QAConfig() defaults).
qa_A = run_qa(
    ds,
    features=feat.features_df,
    out_dir="/tmp/mm_pipeline_demo_A",
    run_tag="dp",
    overwrite=True,
)
tracks_A = qa_A.tracks_by_dataset[ds.dataset_id]
divs_A = qa_A.divisions_by_dataset[ds.dataset_id]
print(f"Reconstructed {tracks_A['track_id'].nunique()} tracks "
      f"across {tracks_A['t'].nunique()} frames, with {len(divs_A)} divisions.")

In [ ]:
# Mother-lineage length time series.
series_A, branch_A, _ = plot_mother_length_series(
    tracks_A, divs_A, ds, title_suffix=" — Example A (DP baseline)",
)

In [ ]:
# Full lineage swimlane: every track's spatial trajectory, with the mother
# lineage highlighted in red and division connectors as dashed lines.
plot_lineage_swimlane(
    tracks_A, divs_A, ds,
    mother_branch=branch_A,
    title_suffix=" — Example A (DP baseline)",
)

## 5. Example B — Classifier-driven

Same trench, now with the scorer from §3 driving the within-pair pick. The candidates and features from Example A are reused — they don't depend on the QA config.

In [ ]:
# Apply the trained scorer to the features from Example A.
scored = run_score(feat.features_df, model=scorer)
print(f"Scored: added columns {sorted(set(scored.scored_df.columns) - set(feat.features_df.columns))}")

In [ ]:
# QA with classifier-driven within-pair picks.
qa_B = run_qa(
    ds,
    scored=scored.scored_df,
    qa_config=QAConfig(within_pair_scorer="classifier"),
    out_dir="/tmp/mm_pipeline_demo_B",
    run_tag="cls",
    overwrite=True,
)
tracks_B = qa_B.tracks_by_dataset[ds.dataset_id]
divs_B = qa_B.divisions_by_dataset[ds.dataset_id]
print(f"Reconstructed {tracks_B['track_id'].nunique()} tracks "
      f"across {tracks_B['t'].nunique()} frames, with {len(divs_B)} divisions.")

In [ ]:
# Mother-lineage length time series for the classifier-driven run.
series_B, branch_B, _ = plot_mother_length_series(
    tracks_B, divs_B, ds, title_suffix=" — Example B (classifier)",
)

In [ ]:
# Full lineage swimlane for the classifier-driven run.
plot_lineage_swimlane(
    tracks_B, divs_B, ds,
    mother_branch=branch_B,
    title_suffix=" — Example B (classifier)",
)